In [ ]:
# Cell 1: Environment Setup & Library Imports
try:
    from bayes_opt import BayesianOptimization
    import xgboost
    import catboost
    import lightgbm
    import shap
except ImportError:
    !pip install bayesian-optimization xgboost catboost lightgbm shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
import os
import re
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from bayes_opt import BayesianOptimization

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor, 
                              ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'serif']

for folder in ['Correlation_Plots', 'Performance_Plots', 'Feature_Importance', 'SHAP_Plots', 'predicted_vs_actual_plots']:
    os.makedirs(folder, exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Cell 2: Load Dataset, Applying Process Counting Logic
import re
from collections import Counter

file_path = r"Copy paste as path\WITHOUT Ti.xlsx"  # Update this path to the dataset
df = pd.read_excel(file_path)


df['Heat Treatment'] = df['Heat Treatment'].fillna('Unaged')
df['Heat Treatment'] = df['Heat Treatment'].replace('O', 'Unaged')

max_temps, min_temps, avg_temps, total_steps, extracted_processes_counts = [], [], [], [], []

for ht in df['Heat Treatment'].astype(str):
    if ht == 'Unaged' or ht.strip() == '':
        max_temps.append(20) 
        min_temps.append(20)
        avg_temps.append(20)
        total_steps.append(0)
        extracted_processes_counts.append({'Unaged': 1})
        continue

    steps = ht.split('+')
    total_steps.append(len(steps))
    row_temps, row_processes = [], []
    
    for step in steps:
        p_match = re.search(r'[A-Za-z]+', step)
        if p_match: row_processes.append(p_match.group(0).capitalize())
            
        t_match = re.search(r'\d+', step)
        if t_match: row_temps.append(float(t_match.group(0)))
            
    max_temps.append(np.max(row_temps) if row_temps else 20)
    min_temps.append(np.min(row_temps) if row_temps else 20)
    avg_temps.append(np.mean(row_temps) if row_temps else 20)
        
    extracted_processes_counts.append(dict(Counter(row_processes)))

df['HT_Total_Steps'] = total_steps
df['HT_Max_Temp'] = max_temps
df['HT_Avg_Temp'] = avg_temps

process_df = pd.DataFrame(extracted_processes_counts).fillna(0).astype(int)
process_df.columns = [f"HT_Count_{col}" for col in process_df.columns]

df = pd.concat([df, process_df], axis=1)
df = df.drop(columns=['Heat Treatment'])

print(f"Process Counting Applied (Phases Included)! Dataset now has {df.shape[1]} columns.")

Process Counting Applied (Phases Included)! Dataset now has 50 columns.


In [ ]:
# Cell 3: Data Cleaning & Feature Selection
metadata_cols = ['Unique ID', 'Material Name', 'Source URL', 'Heat Treatment']
data = df.drop(columns=[col for col in metadata_cols if col in df.columns], errors='ignore')

target_col = 'Thermal Conductivity'

if 'Temperature' in data.columns:
    data['Temperature'] = pd.to_numeric(data['Temperature'], errors='coerce').fillna(20.0)

data = data.fillna(0)

for col in data.columns:
    if col != 'Temperature': 
        data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

data = data[data[target_col] > 0]

print(f"Data cleaning complete. Remaining purely numeric features: {data.columns.tolist()}")

Data cleaning complete. Remaining purely numeric features: ['Alpha', 'Beta', 'Alpha-Beta', 'Temperature', 'Thermal Conductivity', 'Titanium, Ti', 'Copper, Cu', 'Nickel, Ni', 'Aluminum, Al', 'Carbon, C', 'Chromium, Cr', 'Hydrogen, H', 'Iron, Fe', 'Manganese, Mn', 'Molybdenum, Mo', 'Niobium, Nb (Columbium, Cb)', 'Nitrogen, N', 'O + 2N', 'Oxygen, O', 'Silicon, Si', 'Tantalum, Ta', 'Tin, Sn', 'Vanadium, V', 'Zirconium, Zr', 'Palladium, Pd', 'Boron, B', 'Sulfur, S', 'Yttrium, Y', 'Bismuth, Bi', 'O2', 'HT_Total_Steps', 'HT_Max_Temp', 'HT_Avg_Temp', 'HT_Count_Unaged', 'HT_Count_An', 'HT_Count_Bts', 'HT_Count_St', 'HT_Count_Ag', 'HT_Count_Qn', 'HT_Count_Ac', 'HT_Count_Bst', 'HT_Count_Bht', 'HT_Count_Ht', 'HT_Count_Lowtemp', 'HT_Count_Sta', 'HT_Count_Bf', 'HT_Count_Sr']


In [21]:
# Cell 4: Sanitize Column Names
def clean_column_names(columns):
    new_cols = []
    for col in columns:
        clean_name = re.sub(r'[,() +]', '_', col)
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        new_cols.append(clean_name)
    return new_cols

data.columns = clean_column_names(data.columns)
target_col = 'Thermal_Conductivity'

print("Sanitized Column Names:")
print(data.columns.tolist())

Sanitized Column Names:
['Alpha', 'Beta', 'Alpha-Beta', 'Temperature', 'Thermal_Conductivity', 'Titanium_Ti', 'Copper_Cu', 'Nickel_Ni', 'Aluminum_Al', 'Carbon_C', 'Chromium_Cr', 'Hydrogen_H', 'Iron_Fe', 'Manganese_Mn', 'Molybdenum_Mo', 'Niobium_Nb_Columbium_Cb', 'Nitrogen_N', 'O_2N', 'Oxygen_O', 'Silicon_Si', 'Tantalum_Ta', 'Tin_Sn', 'Vanadium_V', 'Zirconium_Zr', 'Palladium_Pd', 'Boron_B', 'Sulfur_S', 'Yttrium_Y', 'Bismuth_Bi', 'O2', 'HT_Total_Steps', 'HT_Max_Temp', 'HT_Avg_Temp', 'HT_Count_Unaged', 'HT_Count_An', 'HT_Count_Bts', 'HT_Count_St', 'HT_Count_Ag', 'HT_Count_Qn', 'HT_Count_Ac', 'HT_Count_Bst', 'HT_Count_Bht', 'HT_Count_Ht', 'HT_Count_Lowtemp', 'HT_Count_Sta', 'HT_Count_Bf', 'HT_Count_Sr']


In [ ]:
# Cell 6: Multicollinearity Check & Dimensionality Reduction (Addressing Reviewers 3 & 4)
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = data.drop(columns=['Thermal_Conductivity'], errors='ignore')

X_vif = X_vif.select_dtypes(include=['number', 'bool']).astype(float)

X_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
vif_data = vif_data[vif_data["Feature"] != "const"].sort_values(by="VIF", ascending=False)

print("--- BEFORE DROPPING FEATURES ---")
print(f"Total numeric features: {X_vif.shape[1]}")
print(f"Feature List: {list(X_vif.columns)}\n")
print("Initial VIF Values (Top 10):")
print(vif_data.head(10)) 

features_to_drop = [
    'HT_Total_Steps',                                 
    'O_2N', 'O2',                                     
    'Boron_B', 'Yttrium_Y',                           
    'Bismuth_Bi', 'Sulfur_S', 'Tantalum_Ta', 'Beta',
    'Titanium_Ti'  #  REMOVING Ti
] 

X_reduced = X_vif.drop(columns=features_to_drop, errors='ignore')

print("\n--- AFTER DROPPING FEATURES ---")
print(f"Total features: {X_reduced.shape[1]}")
print(f"Feature List: {list(X_reduced.columns)}\n")

X_reduced_const = sm.add_constant(X_reduced)
vif_reduced = pd.DataFrame()
vif_reduced["Feature"] = X_reduced_const.columns
vif_reduced["VIF"] = [variance_inflation_factor(X_reduced_const.values, i) for i in range(X_reduced_const.shape[1])]

print("VIF Values after dropping collinear and zero-contribution features (Top 10):")
print(vif_reduced[vif_reduced["Feature"] != "const"].sort_values(by="VIF", ascending=False).head(10))

data = data.drop(columns=features_to_drop, errors='ignore')
print(f"\nFinal reduced dataset shape: {data.shape}")

--- BEFORE DROPPING FEATURES ---
Total numeric features: 46
Feature List: ['Alpha', 'Beta', 'Alpha-Beta', 'Temperature', 'Titanium_Ti', 'Copper_Cu', 'Nickel_Ni', 'Aluminum_Al', 'Carbon_C', 'Chromium_Cr', 'Hydrogen_H', 'Iron_Fe', 'Manganese_Mn', 'Molybdenum_Mo', 'Niobium_Nb_Columbium_Cb', 'Nitrogen_N', 'O_2N', 'Oxygen_O', 'Silicon_Si', 'Tantalum_Ta', 'Tin_Sn', 'Vanadium_V', 'Zirconium_Zr', 'Palladium_Pd', 'Boron_B', 'Sulfur_S', 'Yttrium_Y', 'Bismuth_Bi', 'O2', 'HT_Total_Steps', 'HT_Max_Temp', 'HT_Avg_Temp', 'HT_Count_Unaged', 'HT_Count_An', 'HT_Count_Bts', 'HT_Count_St', 'HT_Count_Ag', 'HT_Count_Qn', 'HT_Count_Ac', 'HT_Count_Bst', 'HT_Count_Bht', 'HT_Count_Ht', 'HT_Count_Lowtemp', 'HT_Count_Sta', 'HT_Count_Bf', 'HT_Count_Sr']

Initial VIF Values (Top 10):
             Feature  VIF
34       HT_Count_An  inf
35      HT_Count_Bts  inf
38       HT_Count_Qn  inf
37       HT_Count_Ag  inf
36       HT_Count_St  inf
39       HT_Count_Ac  inf
43  HT_Count_Lowtemp  inf
44      HT_Count_Sta  inf
4

In [ ]:
# Cell 7: Creating Metallurgical Alloy Families (Zero-Leakage Grouping)
import pandas as pd

print("Classifying alloys into metallurgical families to prevent data leakage...")

def assign_alloy_family(row):
    al = row.get('Aluminum_Al', 0)
    v = row.get('Vanadium_V', 0)
    mo = row.get('Molybdenum_Mo', 0)
    sn = row.get('Tin_Sn', 0)
    zr = row.get('Zirconium_Zr', 0)
    nb = row.get('Niobium_Nb_Columbium_Cb', 0)
    ni = row.get('Nickel_Ni', 0)
    cr = row.get('Chromium_Cr', 0)
    pd_el = row.get('Palladium_Pd', 0)
    cu = row.get('Copper_Cu', 0)
    fe = row.get('Iron_Fe', 0)

    # 1. Nitinol (Shape Memory)
    if ni > 50: return "Nitinol (Ti-Ni Base)"
    
    # 2. Ti-Nb & Medical Beta Alloys
    if nb > 40: return "Ti-Nb Base Alloy"
    if zr > 30 and nb > 10: return "Ti-Zr-Nb Base Alloy"

    # 3. CP Titanium & Minor Alloyed Variants
    if al < 1 and v < 1 and mo < 1 and sn < 1 and nb < 1 and cr < 1 and zr < 1:
        if pd_el > 0.05: return "CP Titanium (Pd-Alloyed)"
        elif cu > 1.5: return "Ti-Cu Alloy"
        else: return "CP Titanium (Standard Grades)"

    # 4. Standard Alpha-Beta & Beta Alloys
    if 5.5 <= al <= 7.5 and 3.5 <= v <= 4.5 and sn < 1 and mo < 1: return "Ti-6Al-4V Family"
    if 5.5 <= al <= 6.5 and 1.5 <= sn <= 2.5 and 3.5 <= zr <= 4.5 and 1.5 <= mo <= 2.5: return "Ti-6Al-2Sn-4Zr-2Mo Family"
    if 2.5 <= al <= 3.5 and 2.0 <= v <= 3.0: return "Ti-3Al-2.5V Family"
    if 4.5 <= al <= 5.5 and 2.0 <= sn <= 3.0 and v < 1: return "Ti-5Al-2.5Sn Family"
    if mo > 14 and 2.5 <= al <= 3.5 and nb > 2: return "Beta-21S Family (Ti-15Mo-3Al-2.7Nb)"
    if v > 14 and cr > 2 and sn > 2 and al > 2: return "Ti-15V-3Cr-3Sn-3Al Family"
    if v > 9 and fe > 1.5 and 2.5 <= al <= 3.5: return "Ti-10V-2Fe-3Al Family"
    if sn > 10 and zr > 4 and 1.5 <= al <= 2.5: return "Ti-11Sn-5Zr-2Al-1Mo Family"

    # 5. Dynamic Fallback for Exotic/Complex Alloys
    major_elements = []
    for col, threshold in [('Aluminum_Al', 1), ('Vanadium_V', 1), ('Molybdenum_Mo', 1), 
                           ('Tin_Sn', 1), ('Zirconium_Zr', 1), ('Niobium_Nb_Columbium_Cb', 1), 
                           ('Chromium_Cr', 1), ('Iron_Fe', 1), ('Manganese_Mn', 1)]:
        if row.get(col, 0) >= threshold:
            major_elements.append(col.split('_')[0])
            
    if major_elements:
        return "Ti-" + "-".join(major_elements) + " Complex Alloy"
    else:
        return "Other Titanium Alloys"

# Apply the metallurgical rules 
data['Composition_Group'] = data.apply(assign_alloy_family, axis=1)

# Generate Summary
family_summary = data['Composition_Group'].value_counts().reset_index()
family_summary.columns = ['Alloy Family', 'Number of Samples']
print(f"Total data points: {len(data)}")
print(f"Number of distinct alloy families (Groups): {data['Composition_Group'].nunique()}\n")
print(family_summary.to_string(index=False))

# Setup the split variables for Nested GroupKFold
X = data.drop(columns=['Thermal_Conductivity', 'Composition_Group'], errors='ignore')
y = data['Thermal_Conductivity']
groups = data['Composition_Group']

Classifying alloys into metallurgical families to prevent data leakage...
Total data points: 309
Number of distinct alloy families (Groups): 40

                                                                               Alloy Family  Number of Samples
                                                              CP Titanium (Standard Grades)                 52
                                                                  Ti-15V-3Cr-3Sn-3Al Family                 24
                                           Ti-Aluminum-Vanadium-Tin-Zirconium Complex Alloy                 22
                                                        Beta-21S Family (Ti-15Mo-3Al-2.7Nb)                 22
                                                                         Ti-3Al-2.5V Family                 17
                                         Ti-Aluminum-Molybdenum-Tin-Zirconium Complex Alloy                 16
                                                Ti-Aluminum-Vanadium-Chromium 

In [ ]:
# CELL 8 (Patched): Nested GroupKFold 
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from skopt import BayesSearchCV

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge

print("Starting Nested GroupKFold Validation...\n")

SEED = 80

models_to_test = {
    "XGB": XGBRegressor(random_state=SEED),
    "CB": CatBoostRegressor(silent=True, random_state=SEED, allow_writing_files=False),
    "LGBM": LGBMRegressor(random_state=SEED, verbose=-1),
    "RF": RandomForestRegressor(random_state=SEED),
    "ET": ExtraTreesRegressor(random_state=SEED),
    "GBR": GradientBoostingRegressor(random_state=SEED),
    "AB": AdaBoostRegressor(random_state=SEED),
    "DT": DecisionTreeRegressor(random_state=SEED),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Ridge": Ridge(),
    "LR": LinearRegression()
}

search_spaces = {
    "XGB": {'max_depth': (3, 10), 'learning_rate': (0.01, 0.3, 'log-uniform'), 'n_estimators': (50, 300)},
    "CB": {'depth': (4, 8), 'learning_rate': (0.01, 0.3, 'log-uniform'), 'iterations': (50, 300)},
    "LGBM": {'max_depth': (3, 10), 'learning_rate': (0.01, 0.3, 'log-uniform'), 'n_estimators': (50, 300)},
    "RF": {'max_depth': (5, 20), 'n_estimators': (50, 300)},
    "ET": {'max_depth': (5, 20), 'n_estimators': (50, 300)},
    "GBR": {'max_depth': (3, 10), 'learning_rate': (0.01, 0.3, 'log-uniform'), 'n_estimators': (50, 300)},
    "AB": {'learning_rate': (0.01, 1.0, 'log-uniform'), 'n_estimators': (50, 300)},
    "DT": {'max_depth': (3, 20)},
    "KNN": {'n_neighbors': (3, 15)},
    "SVR": {'C': (0.1, 100.0, 'log-uniform'), 'gamma': (0.001, 1.0, 'log-uniform')},
    "Ridge": {'alpha': (0.1, 100.0, 'log-uniform')},
    "LR": {}
}

outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=4)

nested_detailed_results = []

oof_predictions = {}
outer_fold_assignments = {}
outer_best_params = []

for name, base_model in models_to_test.items():
    print(f"Evaluating {name} with Nested GroupKFold...")

    outer_r2, outer_rmse, outer_mae = [], [], []
    model_oof = np.full(len(y), np.nan)
    model_fold = np.full(len(y), -1)

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups=data['Composition_Group'])):
        X_train_outer, X_test_outer = X.iloc[train_idx], X.iloc[test_idx]
        y_train_outer, y_test_outer = y.iloc[train_idx], y.iloc[test_idx]
        groups_train_outer = data['Composition_Group'].iloc[train_idx]

        if name in ['KNN', 'SVR', 'Ridge', 'LR']:
            current_model = make_pipeline(StandardScaler(), clone(base_model))
            current_search_space = {f"{base_model.__class__.__name__.lower()}__{k}": v for k, v in search_spaces[name].items()}
        else:
            current_model = clone(base_model)
            current_search_space = search_spaces[name]

        if len(current_search_space) > 0:
            bayes_search = BayesSearchCV(
                estimator=current_model,
                search_spaces=current_search_space,
                cv=inner_cv,
                n_iter=15,
                scoring="r2",  
                random_state=SEED,
                n_jobs=-1
            )
            bayes_search.fit(X_train_outer, y_train_outer, groups=groups_train_outer)
            best_fold_model = bayes_search.best_estimator_
            best_params = bayes_search.best_params_
        else:
            best_fold_model = clone(current_model)
            best_fold_model.fit(X_train_outer, y_train_outer)
            best_params = {}

    
        outer_best_params.append({"Model": name, "Fold": fold_id + 1, **best_params})

        preds = best_fold_model.predict(X_test_outer)

    
        model_oof[test_idx] = preds
        model_fold[test_idx] = fold_id

        outer_r2.append(r2_score(y_test_outer, preds))
        outer_rmse.append(np.sqrt(mean_squared_error(y_test_outer, preds)))
        outer_mae.append(mean_absolute_error(y_test_outer, preds))

    oof_predictions[name] = model_oof
    outer_fold_assignments[name] = model_fold

    mean_r2 = np.mean(outer_r2)
    std_r2 = np.std(outer_r2)

    nested_detailed_results.append({
        "Model": name,
        "Mean_R2": mean_r2,                      
        "SD_R2": std_r2,
        "Mean_RMSE": np.mean(outer_rmse),
        "Mean_MAE": np.mean(outer_mae),
        "Nested Mean R²": f"{mean_r2:.4f} ± {std_r2:.4f}",   
        "Nested Mean RMSE": f"{np.mean(outer_rmse):.4f}",
        "Nested Mean MAE": f"{np.mean(outer_mae):.4f}",
        "Raw_R2_Folds": outer_r2
    })

df_nested_results = pd.DataFrame(nested_detailed_results).sort_values(by="Mean_R2", ascending=False)  
df_outer_best_params = pd.DataFrame(outer_best_params)

print("\n--- NESTED GROUPKFOLD INTERNAL VALIDATION RESULTS ---")
print(df_nested_results[['Model', 'Nested Mean R²', 'Nested Mean RMSE', 'Nested Mean MAE']].to_string(index=False))



df_nested_results_clean = df_nested_results[['Model', 'Nested Mean R²', 'Nested Mean RMSE', 'Nested Mean MAE']]
df_nested_results_clean.to_excel("Table_Internal_Validation_Results.xlsx", index=False)


df_outer_best_params.to_excel("Table_1_Optimal_Hyperparameters.xlsx", index=False)

print("\nSuccess! The performance metrics and hyperparameter tables have been saved as Excel files in your folder.")

Starting Nested GroupKFold Validation...

Evaluating XGB with Nested GroupKFold...
Evaluating CB with Nested GroupKFold...
Evaluating LGBM with Nested GroupKFold...
Evaluating RF with Nested GroupKFold...
Evaluating ET with Nested GroupKFold...
Evaluating GBR with Nested GroupKFold...
Evaluating AB with Nested GroupKFold...
Evaluating DT with Nested GroupKFold...
Evaluating KNN with Nested GroupKFold...
Evaluating SVR with Nested GroupKFold...
Evaluating Ridge with Nested GroupKFold...
Evaluating LR with Nested GroupKFold...

--- NESTED GROUPKFOLD INTERNAL VALIDATION RESULTS ---
Model       Nested Mean R² Nested Mean RMSE Nested Mean MAE
   CB      0.4415 ± 0.5924           2.7631          2.0986
 LGBM      0.4329 ± 0.4174           2.8574          2.1684
   RF      0.3593 ± 0.8074           2.8433          1.9484
   ET      0.3519 ± 0.9398           2.6304          1.8866
  GBR      0.3494 ± 0.7636           2.9221          2.1081
  XGB      0.2922 ± 0.7319           3.0655          2

In [ ]:
# Cell 9: Save R2, RMSE, MAE Results to Excel File
import pandas as pd


if 'df_nested_results' in locals():
    
    
    columns_to_save = ['Model', 'Nested Mean R²', 'Nested Mean RMSE', 'Nested Mean MAE']
    df_export = df_nested_results[columns_to_save].copy()
    
    
    df_export.columns = ['Model', 'R² (Mean ± Std)', 'RMSE (Mean)', 'MAE (Mean)']
    
    
    file_name = "Internal_Validation_Without_Ti.xlsx"
    df_export.to_excel(file_name, index=False)
    
    print(f"Success! Your R², RMSE, and MAE metrics have been saved to '{file_name}'.")
    
else:
    print("Error: The results were not found. Please make sure you run Cell 11 completely before running this cell.")

Success! Your R², RMSE, and MAE metrics have been saved to 'Internal_Validation_Without_Ti.xlsx'.
